In [56]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim

In [57]:
batch_size = 16
context = 32
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embd = 64
n_head = 4
n_layer = 4

In [58]:
torch.manual_seed(1337)
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [59]:
characters = sorted(list(set(text)))
vocab_size = len(characters)
stoi = {s:i for i, s in enumerate(characters)}
itos = {i:s for i, s in enumerate(characters)}
encode = lambda x:[stoi[i] for i in x]
decode = lambda x: "".join([itos[i] for i in x])

In [60]:
data = torch.tensor(encode(text))
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context, (context,))
    x = torch.stack([data[i:i+context] for i in ix])
    y = torch.stack([data[i+1:i+context+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def evaluate():
    out = {}
    model.eval()
    for split in ['train', 'validation']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [61]:
class Head(nn.Module):
    def __init__(self, headsize):
        super().__init__()
        self.headsize = headsize
        self.Query = nn.Linear(n_embd, headsize, bias=False)
        self.Key = nn.Linear(n_embd, headsize, bias=False)
        self.Value = nn.Linear(n_embd, headsize, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(context, context)))
        self.dropout = nn.Dropout(0.2)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.Query(x), self.Key(x), self.Value(x)
        # w = (q @ k.tranpose(-2, -1))/torch.sqrt(self.headsize)
        w = q @ k.transpose(-2, -1) / (self.headsize ** 0.5)
        w = w.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        w = torch.softmax(w, dim=-1)
        output = w @v
        return output

In [62]:
class MultiheadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.heads = nn.ModuleList([Head(headsize=16) for _ in range(n_head)])
        self.linear = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = torch.cat([h(x) for h in self.heads], dim=-1)
        x = self.dropout(self.linear(x))
        return x

In [63]:
class FeedForward_Network(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(n_embd, 4*n_embd)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(4*n_embd, n_embd)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        return self.linear2(self.dropout(self.relu(self.linear1(x))))


In [64]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.MH = MultiheadAttention()
        self.FF = FeedForward_Network()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.MH(self.ln1(x))
        x = x + self.FF(self.ln2(x))
        return x

In [65]:
class Transformers(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.positional_emb = nn.Embedding(context, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln1 = nn.LayerNorm(n_embd)
        self.linear = nn.Linear(n_embd, vocab_size)

    def forward(self, x, targets=None):
        B, T = x.shape
        token = self.token_emb(x)
        positional = self.positional_emb(torch.arange(T, device=device))
        x = self.blocks(token+positional)
        x = self.ln1(x)
        logits = self.linear(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_tokens):
        for _ in range(max_tokens):
            idx_cont = idx[:, -context:]
            logits , loss = self(idx_cont)
            x = logits[:, -1, :]
            probs = F.softmax(x, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [66]:
model = Transformers()
m = model.to(device)

In [67]:
optimizer = optim.Adam(m.parameters(), lr=1e-3)
loss_function = nn.CrossEntropyLoss()
for iter in range(5000):
    if iter % 100 == 0 or iter == 5000 - 1:
        losses = evaluate()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['validation']:.4f}")
    x , y = get_batch('train')
    predictions, loss = m(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

step 0: train loss 4.4091, val loss 4.4025
step 100: train loss 2.6025, val loss 2.6077
step 200: train loss 2.4333, val loss 2.4362
step 300: train loss 2.3396, val loss 2.3517
step 400: train loss 2.2645, val loss 2.2712
step 500: train loss 2.1907, val loss 2.2080
step 600: train loss 2.1433, val loss 2.1574
step 700: train loss 2.1005, val loss 2.1366
step 800: train loss 2.0611, val loss 2.0954
step 900: train loss 2.0375, val loss 2.0813
step 1000: train loss 2.0040, val loss 2.0578
step 1100: train loss 1.9790, val loss 2.0314
step 1200: train loss 1.9554, val loss 2.0213
step 1300: train loss 1.9340, val loss 2.0052
step 1400: train loss 1.9122, val loss 1.9931
step 1500: train loss 1.9078, val loss 1.9825
step 1600: train loss 1.8883, val loss 1.9657
step 1700: train loss 1.8691, val loss 1.9449
step 1800: train loss 1.8556, val loss 1.9540
step 1900: train loss 1.8481, val loss 1.9555
step 2000: train loss 1.8335, val loss 1.9376
step 2100: train loss 1.8300, val loss 1.9323


In [68]:
startpoint = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(startpoint, max_tokens=2000)[0].tolist()))



As little as the santured fait, cape I would what respenthy's
Friblong, unsay! To blues myrach;
Why supper my gost That donger,
That our coustions: no, not morchiolous in and, but doud
Bronary, grring roudd him, what her toar honount,
If to she dignown whold enfell be thou world inpoin?

First:
Hound your too gropther: fawn. Loks?
Firh's joy, fore; it, what it loes you; what a shall was yourely hip to apper
Heirs-an moucane prown ours.
The mont: but have lame thoughts abritt, faidly his limender in?
Herether beque of your I king to patceraths's,
No! 'ther God no do hersef for be youngelity to my canspeld;
I'll Cleadneny is riangured we lies have; fortor is not his trongs.

SOMERD:
Whot to wookevent, but seech, queet to withis ank to upornsonhfres,
For migstannown Ise is aredienuon,
es of Catusperet, care her seiking one your to blookly tuie!

MARK:
We thee, to for this watch ove patiinour could you fait; her?
And, over but, and good Coapleechs,
That the Louchiand and wold mean,
Pans! 

56243